# Bangla RAG question answering with a fine-tuned BanglaT5: Colab runner

Runtime: **Runtime > Change runtime type > L4 GPU** (High-RAM optional).
Run the cells top to bottom. Outputs and checkpoints are saved to Google Drive so a disconnect loses nothing.

**Status:** Step 0 (setup and token-length check) is ready. Training and evaluation cells are added in later steps.

## 1. Get the code and connect Drive

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')

REPO_URL = "https://github.com/mehediparvez/bangla_rag_evaluation.git"
BRANCH = "bangla-t5-rag-qa"
REPO_DIR = "/content/bangla_rag_evaluation"

if not os.path.exists(REPO_DIR):
    !git clone -b {BRANCH} {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} fetch origin && git -C {REPO_DIR} checkout {BRANCH} && git -C {REPO_DIR} pull origin {BRANCH}
%cd {REPO_DIR}

# Everything worth keeping goes here (survives a Colab disconnect)
OUT_DIR = "/content/drive/MyDrive/bangla_t5_rag"
os.makedirs(OUT_DIR, exist_ok=True)
print("Saving outputs to", OUT_DIR)

## 2. Install dependencies and check the GPU

In [ ]:
!pip -q install transformers sentencepiece
!pip -q install git+https://github.com/csebuetnlp/normalizer
!nvidia-smi

## 3. Download BanglaRQA
The dataset is fetched at run time and is not stored in the repository (CC BY-NC-SA 4.0).

In [ ]:
!python -m bangla_rag.data

## 4. Step 0: are whole passages too long for BanglaT5?
BanglaT5 reads at most 512 tokens. This counts how many passages are longer.
Send the printed table back before going further: it decides whether whole passages can be the retrieval unit.

In [ ]:
!python scripts/check_token_lengths.py

## Next steps (cells added later)
5. Fine-tune BanglaT5 on the train split (gold passage as context, unanswerable included), checkpoints to Drive
6. Build BM25, dense and hybrid retrievers, compute hit rates
7. Evaluate: gold, BM25, dense, hybrid (and a no-passage extra) on the 1,493 test questions
8. McNemar tests with Holm correction, results table, error-analysis CSV